# Stage 3 — comma2k19 HF demo V1

공식 Hugging Face demo 64개를 10Hz로 변환하고, route-group holdout에서 기존 baseline과 후보 모델을 비교한 뒤 제출 ZIP을 두 번 smoke 검증합니다. 셀을 위에서 아래로 한 번씩 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import os, shutil, subprocess, sys, json

DRIVE_ROOT = Path('/content/drive/MyDrive/블랙박스 영상 기반 사고 분석')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive가 실제로 마운트되지 않았습니다.'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BASE_BUNDLE = DRIVE_ROOT / 'dacon236753_colab_bundle.zip'  # 최초 1회만 업로드
PATCH_BUNDLE = DRIVE_ROOT / 'dacon236753_patch.zip'        # 이후 작은 수정은 이것만 교체
PROJECT_ROOT = Path('/content/dacon236753')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / 'model/stage3/best.pt').is_file():
    assert BASE_BUNDLE.is_file(), f'기본 번들이 없습니다: {BASE_BUNDLE}'
    shutil.unpack_archive(str(BASE_BUNDLE), str(PROJECT_ROOT))
assert PATCH_BUNDLE.is_file(), f'최신 patch가 없습니다: {PATCH_BUNDLE}'
shutil.unpack_archive(str(PATCH_BUNDLE), str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
assert (PROJECT_ROOT / 'tools/fetch_stage3_hf.py').is_file()
patch_info = json.loads((PROJECT_ROOT / 'PATCH_INFO.json').read_text())
print('project:', PROJECT_ROOT)
print('base ready:', (PROJECT_ROOT / 'model/stage3/best.pt').is_file())
print('patch:', patch_info['created_utc'])

In [ ]:
import torch, torchvision, cv2, pandas as pd
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'cv2:', cv2.__version__)
!python tools/preflight.py

In [ ]:
EXPERIMENT = 's3_hf_demo_v1'
RAW_ROOT = DRIVE_ROOT / 'external_data/stage3/raw/comma2k19_hf_demo'
DATA_ROOT = DRIVE_ROOT / 'external_data/stage3'
ARTIFACT = DRIVE_ROOT / 'experiments' / EXPERIMENT
for folder in ('config','checkpoints','features','predictions','reports','submission','logs'):
    (ARTIFACT / folder).mkdir(parents=True, exist_ok=True)

parquet_files = sorted((RAW_ROOT / 'data').glob('*.parquet'))
video_zip = RAW_ROOT / 'compression_challenge/test_videos.zip'
if len(parquet_files) != 3 or not video_zip.is_file() or video_zip.stat().st_size <= 2_000_000_000:
    print('Drive 원본이 없거나 불완전합니다. 로컬 캐시를 거쳐 Drive에 실제 파일로 복구합니다.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'], check=True)
    subprocess.run([sys.executable, str(PROJECT_ROOT / 'tools/fetch_stage3_hf.py'), '--raw-root', str(RAW_ROOT)], check=True)

parquet_files = sorted((RAW_ROOT / 'data').glob('*.parquet'))
video_zip = RAW_ROOT / 'compression_challenge/test_videos.zip'
assert len(parquet_files) == 3, f'Drive parquet 파일은 {len(parquet_files)}개입니다: {parquet_files}'
assert video_zip.is_file() and video_zip.stat().st_size > 2_000_000_000, f'Drive video ZIP이 없습니다/불완전합니다: {video_zip}'
assert all(not path.is_symlink() for path in parquet_files + [video_zip]), 'Drive 원본에 symlink가 남아 있습니다.'
PARENT_CHECKPOINT = ARTIFACT / 'checkpoints/parent_baseline.pt'
if not PARENT_CHECKPOINT.is_file():
    shutil.copy2(PROJECT_ROOT / 'model/stage3/best.pt', PARENT_CHECKPOINT)
print('experiment:', ARTIFACT)
print('raw files:', len(parquet_files), 'parquet +', round(video_zip.stat().st_size / 1e9, 3), 'GB zip')
print('frozen parent:', PARENT_CHECKPOINT)

## 1. HF 원본 → 10Hz 데이터

첫 실행은 64개 HEVC를 `/content`에 풀고 10Hz MP4로 변환하므로 시간이 걸립니다. 결과는 Drive의 `external_data/stage3/processed/`에 보존되고 재실행 시 기존 MP4는 건너뜁니다.

In [ ]:
!python tools/prepare_stage3_hf.py --raw-root "$RAW_ROOT" --data-root "$DATA_ROOT" --work-dir /content/comma2k19_hf_work

In [ ]:
manifest = pd.read_csv(DATA_ROOT / 'manifest.csv')
labels = pd.read_csv(DATA_ROOT / manifest.iloc[0].labels_path)
train_routes = set(manifest.loc[manifest.split == 'train', 'route_group'])
valid_routes = set(manifest.loc[manifest.split == 'validation', 'route_group'])
assert not (train_routes & valid_routes), 'route leakage'
assert manifest.id.is_unique and {'train','validation'} == set(manifest.split)
print(manifest.groupby('split').agg(videos=('id','size'), routes=('route_group','nunique'), samples=('samples','sum')))
print('\naccel\n', labels.accel_label.value_counts())
print('\nsteer\n', labels.steer_label.value_counts())
print('PASS: manifest, route split, labels')

## 2. 기존 MViTv2 특징으로 Stage 3 head 학습

baseline backbone은 고정하고 accel/steer head만 학습합니다. 최초 실행은 특징을 계산해 Drive에 저장하며, 이후에는 저장된 특징을 재사용합니다.

In [ ]:
!python tools/train_stage3_hf.py --data-root "$DATA_ROOT" --experiment-dir "$ARTIFACT" --baseline-checkpoint "$PARENT_CHECKPOINT" --epochs 80 --feature-stride 2 --batch-size 6

In [ ]:
summary = json.loads((ARTIFACT / 'reports/training_summary.json').read_text())
print(json.dumps(summary, indent=2))
assert (ARTIFACT / 'checkpoints/stage3_best.pt').is_file()
print('PASS: baseline/candidate local metrics and compatible checkpoint')

## 3. 제출 ZIP 생성 및 두 번 smoke

후보 Stage 3만 교체하고 Stage 1·2는 공식 baseline을 그대로 유지합니다. 두 추론 중 하나라도 실패하면 smoke marker가 생성되지 않습니다.

In [ ]:
!python tools/package_stage3_candidate.py --project-root "$PROJECT_ROOT" --experiment-dir "$ARTIFACT" --checkpoint "$ARTIFACT/checkpoints/stage3_best.pt"

In [ ]:
# FINAL SUBMISSION GATE — 이 셀은 항상 마지막에 둡니다.
gate = [sys.executable, str(PROJECT_ROOT / 'tools/go_no_go.py'), '--stage', 'stage3', '--baseline-local', str(ARTIFACT / 'reports/baseline_local_metrics.json'), '--candidate-local', str(ARTIFACT / 'reports/local_metrics.json'), '--evaluation-kind', 'external_group_holdout', '--json-out', str(ARTIFACT / 'reports/submission_gate.json')]
if (ARTIFACT / 'reports/zip_smoke_pass.json').is_file(): gate.append('--smoke-pass')
subprocess.run(gate, check=False)